# Benchmarks, analysis and inspection of the OPFData task and datasets

Given the state and operating conditions of an electric power system at a single point in time, the goal of the OPFData task is to solve instances of the Alternating Current Optimal Power Flow (ACOPF) problem. Specifically, the objective is to determine the most cost-efficient power generation dispatch while satisfying the physical and operational constraints governing the power network.

The ACOPF problem is a non-convex and NP-hard optimization problem that is traditionally solved using physics-based optimization techniques employed by power system operators. Improving the speed, scalability, and solution quality of ACOPF through ML-based approaches is a requirement for enabling both the planning and real-time operation of highly renewable power systems, which are expected to face significantly greater variability and uncertainty in electricity supply than today’s predominantly fossil fuel-based systems.

ACOPF problem instances are naturally represented as graphs, where nodes correspond to buses that can be connected to one or more electricity generator, and edges represent either transmission lines or transformers within the power grid.

The OPFData task can be formulated as either a supervised or an unsupervised ML problem.

In the supervised learning setting, the datasets include optimal solutions obtained from conventional non-ML optimization solvers for each problem instance. These solutions can be used as target labels for supervised objectives such as the root mean squared error (RMSE) loss.

In the unsupervised learning setting, we provide the mathematical expressions defining the ACOPF objective function together with the associated equality and inequality constraints in a cannonical way. The resulting objective and constraint violation terms can be combined to construct a physics-informed unsupervised learning loss.

We define six sub-task datasets designed to evaluate the generalization capabilities of solution methods across different power system topologies and scales. The complete dataset consists of problem instances derived from ten distinct power grid topologies, with training, validation, and testing splits constructed according to topology size categories. Each sub-task trains and validates models on one group of topologies (small, medium, or large-scale systems), while evaluating performance on a different group. This setup enables a systematic assessment of cross-topology generalization, scalability, and transferability of ML methods. 

## Overview
---
1. Creating benchmarks
2. Analysing characteristics
3. Inspecting aigrids data
4. Inspecting processed data
5. Inspecting raw data

In [ ]:
import json
import os
import math

import utils
import aigrids

# set the path to your configuration file here.
path_cfg = '../config_arsam.yml'

# load the configuration file and show us
cfg = utils.parse_config(path_cfg)
cfg

## 1. Creating benchmarks

In [ ]:
# TODO

## 2. Analysing characteristics

In [1]:
# TODO

## 3. Inspecting aigrids data

In [ ]:
taskdata = aigrids.load.load_task(
    'OPFData', 
    'train_small_test_medium',
    cfg['path_root'],
    data_frac = 0.01
)

In [ ]:
# TODO

## 4. Inspecting processed data

In [1]:
# Set up base paths
path_raw_base = os.path.join(cfg['path_root'], 'raw', 'OPFData')

In [2]:
def merge_file(grid_type: str, release_type:str, target_fsize_MB=10000, results_path:str, save=False):
    """Merges all data files"""
    # set path to current grid type
    path_gridtype = os.path.join(path_raw_base, release_type, grid_type)
    # read folders in grid type
    group_list = os.listdir(path_gridtype)
    # iterate over elements in group list
    for group in group_list:
        # tell us what's going on
        print(f"Iterating {group}")
        # create path to group
        path_group = os.path.join(path_gridtype, group)
        # read file list in iterated group folder
        file_list = os.listdir(path_group)
        # compute number of files
        n_files_group = len(file_list)
        # tell us the stats
        print(f"The group folder has {n_files_group} files.")
        # read a sample file
        path_sample_json = os.path.join(path_group, file_list[0])
        # little test if sample is json
        if path_sample_json.endswith(".json"):
            # read size of sample file
            sample_fsize_byte = os.path.getsize(path_sample_json)
            target_sample_size = math.ceil(target_fsize_MB / (sample_fsize_byte * 10e-6))
        else:
            raise SampleTypeError("sample file does not end in .json. Check your code!")

        # tell us what's going on
        print(f"Merging {target_sample_size} samples into one file.")
        print(f"This will result in {math.ceil(n_files_group/target_sample_size)} files for this group.")
        # iterate over all files in target_sample_size steps
        fload_counter = 0
        fsave_counter = 0
        dict_size_counter = 0
        save_dict = {}
        while fload_counter < len(file_list):
            # set file name for load
            fname_load = file_list[fload_counter]
            # increment fload_counter
            fload_counter += 1
            # set path for loading file name
            path_load = os.path.join(path_group, fname_load)
            # read the json file
            with open(path_load) as json_file:
                data_dict = json.load(json_file)

            # split the file name 
            fname_dict = fname_load.split('_')[1].split('.')[0]
            # save loaded json to expanding dictionary
            save_dict[fname_dict] = data_dict
            # increment dictionary size counter
            dict_size_counter += 1
            # check if reached target sample size
            if dict_size_counter == target_sample_size or fload_counter == len(file_list):
                print(f"Starting to save file number {fsave_counter+1} with {dict_size_counter} samples")
                # save merged files
                if save:
                    # set filename
                    save_fname = f"merged_{fsave_counter+1}.json"
                    # set path to saving
                    save_path_group = os.path.join(results_path, release_type, grid_type, group) 
                    # create results path if not exists
                    if not os.path.exists(save_path_group):
                        os.makedirs(save_path_group)

                    # set path
                    save_path_file = os.path.join(save_path_group, save_fname) 
                    # save save_dict as json file
                    with open(save_path_file, 'w') as json_save_file:
                        json.dump(save_dict, json_save_file)

                    print(f"Successfuly saved {save_path_file}")
                    
                # increment fname_counter
                fsave_counter += 1
                # reset dict size counter and save_dict
                dict_size_counter = 0
                save_dict = {}
        

In [2]:
# set release type
release_type = "dataset_release_1_nminusone" # "dataset_release_1_nminusone" or "dataset_release_1"

# set results path
results_path = # SET PATH where you want results to be stored.

# call merging function
merge_file("pglib_opf_case2000_goc", release_type, 10000, results_path, save=True)

NameError: name 'os' is not defined

## 5. Inspecting raw data

In [4]:
# Set up paths
path_raw_base = "../data/raw/gridopt-dataset/"
release_type = "dataset_release_1_nminusone"
grid_type = "pglib_opf_case118_ieee"
sample_gname = 'group_0'
sample_fname1 = 'example_1.json'
sample_fname2 = 'example_2.json'

path_sample1 = os.path.join(path_raw_base, release_type, grid_type, sample_gname, sample_fname1)
path_sample2 = os.path.join(path_raw_base, release_type, grid_type, sample_gname, sample_fname2)

# read the json file
with open(path_sample1) as json_file1:
    data_dict1 = json.load(json_file1)

data_dict1

{'grid': {'nodes': {'bus': [[138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [345.0, 2.0, 0.94, 1.06],
    [345.0, 1.0, 0.94, 1.06],
    [345.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [345.0, 2.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [138.0, 1.0, 0.94, 1.06],
    [345.0, 1.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 2.0, 0.94, 1.06],
    [138.0, 1.0,

In [30]:
for key, value in data['grid']['edges'].items():
    print(key)

ac_line
transformer
generator_link
load_link
shunt_link


In [19]:
data['metadata']

{'objective': 2066.454375909309}

In [20]:
data['solution']

{'nodes': {'bus': [[-2.000671981789483e-33, 1.0600000103356157],
   [-0.0988951290232895, 1.0326361403132205],
   [-0.22645175752822816, 1.0105088156331814],
   [-0.18524733704146362, 1.0037836728607352],
   [-0.15915366646911178, 1.0079062291028897],
   [-0.25515981328719517, 1.0600000087190324],
   [-0.2413446898942717, 1.023659409290172],
   [-0.2413446898942717, 1.023659409290172],
   [-0.2709257361584539, 1.0235339170944686],
   [-0.27291289296995347, 1.021968291031788],
   [-0.26574133857535553, 1.0371815996547118],
   [-0.2692047974360478, 1.0439651110913386],
   [-0.26937625266434795, 1.0376138062992242],
   [-0.2862003822595075, 1.0124481198439448]],
  'generator': [[2.608846335607063, 0.044101778483469106],
   [-9.960681129533501e-09, 0.29999999535086597],
   [0.0, 0.3999997867011594],
   [0.0, 0.2140087892294048]]},
 'edges': {'ac_line': {'senders': [0,
    0,
    1,
    1,
    1,
    2,
    3,
    5,
    5,
    5,
    6,
    6,
    8,
    8,
    9,
    11,
    12],
   'rece

In [21]:
data['grid']

{'nodes': {'bus': [[1.0, 3.0, 0.94, 1.06],
   [1.0, 2.0, 0.94, 1.06],
   [1.0, 2.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 2.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 2.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06],
   [1.0, 1.0, 0.94, 1.06]],
  'generator': [[100.0,
    1.7,
    0.0,
    3.4,
    0.05,
    0.0,
    0.1,
    1.0,
    0.0,
    792.0951,
    0.0],
   [100.0, 0.295, 0.0, 0.59, 0.0, -0.3, 0.3, 1.0, 0.0, 2326.9494, 0.0],
   [100.0, 0.0, 0.0, 0.0, 0.2, 0.0, 0.4, 1.0, 0.0, 0.0, 0.0],
   [100.0, 0.0, 0.0, 0.0, 0.09, -0.06, 0.24, 1.0, 0.0, 0.0, 0.0]],
  'load': [[0.2074537468887477, 0.14104720823994257],
   [0.8471011309100136, 0.22282354552053588],
   [0.45028731853800247, -0.03581809810780636],
   [0.08541906111921224, 0.017578741554177255],
   [0.09622104793645124, 0.07907267225231196],
   [0.35274569767886677, 0.1910895443281